# 计算检索和回答指标

RAG 评估需要分开看三件事：找到的资料是否回应问题，回答是否有资料支持，以及回答是否真正回应了问题。这三项常合称 RAG Triad。

三者不能互相代替。找到了正确页面，不等于回答已经正确；回答语句通顺，也不等于有原文支持。本页是固定输入上的指标计算运行示例，不是某种方法的前后效果对照。

In [1]:
import sys
from pathlib import Path

def find_course_root(start):
    for folder in (start, *start.parents):
        if (folder / 'data' / 'dataset/manifest.json').is_file():
            return folder
        nested = folder / 'notebook' / 'C7 高级 RAG 技巧'
        if (nested / 'data' / 'dataset/manifest.json').is_file():
            return nested
    raise FileNotFoundError('请从仓库根、C7 根或本章目录启动')

course_root = find_course_root(Path.cwd().resolve())
if str(course_root) not in sys.path:
    sys.path.insert(0, str(course_root))

from common.eval_utils import precision_at_k, recall_at_k

expected_pages = {18, 21}
returned_pages = [21, 18, 99, 16]
answer_claims = [
    {"text": "宏平均先计算每个类别的指标。", "supported": True, "answers_question": True},
    {"text": "微平均不受类别数量影响。", "supported": False, "answers_question": True},
]

retrieval_flags = [page in expected_pages for page in returned_pages]
context_precision = precision_at_k(retrieval_flags, len(retrieval_flags))
context_recall = recall_at_k(retrieval_flags, len(expected_pages), len(retrieval_flags))
support_rate = sum(item["supported"] for item in answer_claims) / len(answer_claims)
answer_relevance = sum(item["answers_question"] for item in answer_claims) / len(answer_claims)

print(f"上下文精确率：{context_precision:.2f}")
print(f"上下文召回率：{context_recall:.2f}")
print(f"回答结论有原文支持：{support_rate:.2f}")
print(f"回答结论回应了问题：{answer_relevance:.2f}")

上下文精确率：0.50
上下文召回率：1.00
回答结论有原文支持：0.50
回答结论回应了问题：1.00


## 按环节继续拆分

- 检索：必要资料命中、正确资料排名、无关资料数量。
- 回答：必要结论完整度、无原文支持的说法、引用页码正确性。
- 整体流程：检索次数、输入文字量、耗时、费用、拒答和权限错误。

本教程直接计算这些结果。先写清楚想检查什么，再决定记录哪些字段。



## RAG Triad 与分层指标的完整读法

RAG Triad（RAG 三元组）围绕用户问题、检索上下文和回答建立三条关系：上下文相关性问“找回的资料能不能帮助回答问题”；忠实度/有据性问“回答中的每个结论能否回到资料”；答案相关性问“回答是否真正回应问题”。三者不能互相替代：正确页面命中不保证回答有据，回答流畅也不保证切题。

检查时要分开看：检索部分包括正确资料的比例、找全了多少资料、正确资料的排名和无关资料比例；回答部分包括多少结论有原文依据、是否答全、是否切题和引用页是否正确；整套系统还要记录总耗时、检索轮数、输入输出 token、费用、拒答、权限错误和超时。阈值要用自己的问题集确定，不能照抄示例数字。

本页现有输出是固定输入的三项计算示例。下面的代码只演示公式；上面的固定输入示例仍是本页可直接阅读的结果。Recall 和 AP 的分母必须显式传入完整评估集中的相关证据数；若只统计当前命中项，漏召回时会错误地得到 1。


In [2]:
from math import log2
from common.eval_utils import average_precision, precision_at_k, recall_at_k

def ndcg_at_k(relevances, k):
    all_values = list(relevances)
    values = all_values[:k]
    dcg = sum((2 ** rel - 1) / log2(rank + 1) for rank, rel in enumerate(values, 1))
    # IDCG 应从全部候选相关性中取前 k 个，而不是只重排已返回的 k 个。
    ideal = sorted(all_values, reverse=True)[:k]
    idcg = sum((2 ** rel - 1) / log2(rank + 1) for rank, rel in enumerate(ideal, 1))
    return dcg / idcg if idcg else 0.0

def triad_report(retrieval_flags, claim_records, total_relevant):
    """claim_records 的字段为 supported、answers_question；返回可解释的三项比例。"""
    claims = list(claim_records)
    return {
        "context_precision": precision_at_k(retrieval_flags, len(retrieval_flags)),
        "context_recall": recall_at_k(retrieval_flags, total_relevant, len(retrieval_flags)),
        "faithfulness": sum(bool(x.get("supported")) for x in claims) / max(1, len(claims)),
        "answer_relevance": sum(bool(x.get("answers_question")) for x in claims) / max(1, len(claims)),
    }

# 若 expected_pages 只是离线核对标签，它不能在检索前进入 query；上线时应使用人工
# relevance 标注或经过校准的评估集。
demo_report = triad_report([1, 0, 1], [
    {"supported": True, "answers_question": True},
    {"supported": False, "answers_question": True},
], total_relevant=3)
print(
    f"指标示例：前 2 条的上下文精确率 {precision_at_k([1, 0, 1], 2):.2f}，"
    f"前 3 条的上下文召回率 {recall_at_k([1, 0, 1], 3, 3):.2f}，"
    f"AP {average_precision([1, 0, 1], total_relevant=3):.2f}，"
    f"排序质量 {ndcg_at_k([0, 1, 2], 3):.3f}；"
    f"上下文覆盖 {demo_report['context_recall']:.2f}，回答有据性 {demo_report['faithfulness']:.2f}，"
    f"回答相关性 {demo_report['answer_relevance']:.2f}。"
)

指标示例：前 2 条的上下文精确率 0.50，前 3 条的上下文召回率 0.67，AP 0.56，排序质量 0.587；上下文覆盖 0.67，回答有据性 0.50，回答相关性 1.00。


## 为什么这里直接计算指标

本章只需要核对固定问题、必要资料、回答要点和拒答条件，现有代码已经能完成这些检查。额外引入评估框架会增加安装、版本适配和模型调用成本，却不能替代人工写清楚正确答案与必要证据。因此教程保留本地计算和逐题结果，外部框架只在需要时另行接入。

教程中的评估输入统一保留问题编号、问题内容、必要页、回答要点和拒答条件。运行后保存逐题结果，再比较改动前后是否提升，以及有没有把原本正确的问题改坏。


## 本页导航

- 章节入口：[本章 README](README.md)
- 运行准备：[C7 统一运行准备](../README.md#运行准备)
- 相关下一步：[比较改动前后](比较改动前后.ipynb)

